# 🌾 Crop Disease CNN Training — MobileNetV2 Transfer Learning

This notebook trains a CNN model for **Crop Disease Detection** using MobileNetV2 transfer learning.

**Dataset:** Google Drive folder (no local download needed)

### Steps:
1. Mount Google Drive & access dataset
2. Explore dataset structure
3. Train MobileNetV2 CNN (2-phase: frozen → fine-tune)
4. Evaluate & save model
5. Download `crop_disease_model.h5` and `class_labels.json`

⚠️ **IMPORTANT:** Before running, go to **Runtime → Change runtime type → GPU (T4)**

## 1️⃣ Setup & Mount Google Drive

In [ ]:
# Mount Google Drive to access dataset directly
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install gdown for downloading shared Drive folders
!pip install -q gdown

In [ ]:
import os

# ============================================================
# Point to the dataset in your mounted Google Drive.
# The folder contains subfolders: chilli, coffee, paddy
# each with disease class folders inside.
# ============================================================

# Try to find the dataset folder in Drive
DATASET_PATH = None

# Check common paths
possible_paths = [
    '/content/drive/MyDrive/chilli',  # If only chilli was added
    '/content/drive/MyDrive',         # Root of Drive
    '/content/dataset',               # Previously downloaded
]

# Search for the shared folder that contains chilli/coffee/paddy
import glob
for pattern in ['chilli', 'coffee', 'paddy']:
    matches = glob.glob(f'/content/drive/MyDrive/**/{pattern}', recursive=True)
    for m in matches:
        if os.path.isdir(m):
            parent = os.path.dirname(m)
            # Check if parent has at least 2 of the 3 crop folders
            siblings = os.listdir(parent)
            crop_count = sum(1 for s in siblings if s.lower() in ('chilli', 'coffee', 'paddy', 'chili'))
            if crop_count >= 2:
                DATASET_PATH = parent
                break
    if DATASET_PATH:
        break

if not DATASET_PATH:
    # Fallback: check if crops are directly in MyDrive
    drive_root = '/content/drive/MyDrive'
    if os.path.exists(drive_root):
        items = os.listdir(drive_root)
        crop_count = sum(1 for s in items if s.lower() in ('chilli', 'coffee', 'paddy', 'chili'))
        if crop_count >= 2:
            DATASET_PATH = drive_root

if not DATASET_PATH:
    DATASET_PATH = '/content/drive/MyDrive'
    print('⚠️ Could not auto-detect dataset folder.')
    print('Please update DATASET_PATH to the folder containing chilli/, coffee/, paddy/')

print(f'Dataset path: {DATASET_PATH}')
if os.path.exists(DATASET_PATH):
    contents = os.listdir(DATASET_PATH)
    crop_folders = [c for c in contents if c.lower() in ('chilli', 'coffee', 'paddy', 'chili')]
    print(f'Crop folders found: {crop_folders}')
    print(f'All contents: {contents}')

## 2️⃣ Explore Dataset Structure

In [ ]:
import os

# List top-level contents of dataset
print(f'Dataset root: {DATASET_PATH}')

if not os.path.exists(DATASET_PATH):
    print('❌ Dataset path does not exist! Please re-run the download cell above.')
else:
    print(f'Contents: {os.listdir(DATASET_PATH)}')
    print()

    def find_image_dirs(root, depth=0, max_depth=4):
        """Recursively find directories containing images."""
        if depth > max_depth:
            return []
        results = []
        try:
            items = os.listdir(root)
        except PermissionError:
            return []
        
        img_exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')
        has_images = any(f.lower().endswith(img_exts) for f in items if os.path.isfile(os.path.join(root, f)))
        if has_images:
            img_count = sum(1 for f in items if f.lower().endswith(img_exts))
            results.append((root, img_count))
        
        for item in sorted(items):
            full = os.path.join(root, item)
            if os.path.isdir(full):
                results.extend(find_image_dirs(full, depth + 1, max_depth))
        return results

    img_dirs = find_image_dirs(DATASET_PATH)
    print(f'Found {len(img_dirs)} directories with images:')
    total_images = 0
    for d, count in img_dirs:
        rel = os.path.relpath(d, DATASET_PATH)
        print(f'  {rel}: {count} images')
        total_images += count
    print(f'\nTotal images: {total_images}')
    
    if total_images == 0:
        print('\n⚠️ No images found. Check that DATASET_PATH points to the correct folder.')

In [ ]:
# ============================================================
# AUTO-DETECT TRAIN/VAL & BUILD SPLIT
# Scans all subfolders recursively to find image class folders,
# regardless of nesting depth (up to 5 levels).
# ============================================================

import shutil
from sklearn.model_selection import train_test_split

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')

def count_images(folder):
    """Count image files directly inside a folder."""
    try:
        return sum(1 for f in os.listdir(folder) 
                   if f.lower().endswith(IMG_EXTS) and os.path.isfile(os.path.join(folder, f)))
    except:
        return 0

def find_all_image_classes(root, prefix='', depth=0, max_depth=5):
    """Recursively find all leaf directories containing images.
    Returns dict: label -> path"""
    results = {}
    try:
        entries = sorted(os.listdir(root))
    except:
        return results
    
    dirs = [e for e in entries if os.path.isdir(os.path.join(root, e))]
    img_count = count_images(root)
    
    if img_count > 0 and prefix:
        # This folder has images — it's a class folder
        results[prefix] = root
    
    for d in dirs:
        d_path = os.path.join(root, d)
        new_prefix = f'{prefix}_{d}' if prefix else d
        sub_results = find_all_image_classes(d_path, new_prefix, depth + 1, max_depth)
        results.update(sub_results)
    
    return results

contents = os.listdir(DATASET_PATH)
subdirs = [d for d in contents if os.path.isdir(os.path.join(DATASET_PATH, d))]

TRAIN_DIR = None
VAL_DIR = None

# Check for existing train/val split
for name in subdirs:
    lower = name.lower()
    if lower in ('train', 'training'):
        TRAIN_DIR = os.path.join(DATASET_PATH, name)
    elif lower in ('val', 'valid', 'validation', 'test'):
        VAL_DIR = os.path.join(DATASET_PATH, name)

if TRAIN_DIR and VAL_DIR:
    print(f'Found existing train/val split!')
    print(f'  Train: {TRAIN_DIR}')
    print(f'  Val:   {VAL_DIR}')
else:
    print(f'Scanning {DATASET_PATH} for all image class folders...\n')
    
    # Debug: show what's inside each top-level folder
    for d in sorted(subdirs):
        d_path = os.path.join(DATASET_PATH, d)
        inner = os.listdir(d_path)
        inner_dirs = [i for i in inner if os.path.isdir(os.path.join(d_path, i))]
        inner_imgs = count_images(d_path)
        print(f'📁 {d}/ → {inner_imgs} images, {len(inner_dirs)} subdirs: {inner_dirs[:10]}')
        # Show one more level
        for i_d in sorted(inner_dirs)[:5]:
            i_path = os.path.join(d_path, i_d)
            i_imgs = count_images(i_path)
            i_inner = [x for x in os.listdir(i_path) if os.path.isdir(os.path.join(i_path, x))]
            print(f'   📁 {i_d}/ → {i_imgs} images, {len(i_inner)} subdirs: {i_inner[:5]}')
        if len(inner_dirs) > 5:
            print(f'   ... and {len(inner_dirs) - 5} more subdirs')
    print()
    
    # Find ALL image class folders recursively
    all_classes = find_all_image_classes(DATASET_PATH)
    
    print(f'📊 Total classes found: {len(all_classes)}')
    for label in sorted(all_classes):
        img_count = count_images(all_classes[label])
        print(f'  {label}: {img_count} images')
    
    if all_classes:
        TRAIN_DIR = '/content/split_data/train'
        VAL_DIR = '/content/split_data/val'
        
        # Clean previous split
        if os.path.exists('/content/split_data'):
            shutil.rmtree('/content/split_data')
        os.makedirs(TRAIN_DIR, exist_ok=True)
        os.makedirs(VAL_DIR, exist_ok=True)
        
        total_train = 0
        total_val = 0
        
        print('\nCreating 80/20 train/val split...')
        for label, src_path in sorted(all_classes.items()):
            images = [f for f in os.listdir(src_path) if f.lower().endswith(IMG_EXTS)]
            
            if len(images) < 3:
                print(f'  ⚠️ Skipping {label}: only {len(images)} images')
                continue
            
            split_ratio = 0.3 if len(images) < 10 else 0.2
            train_imgs, val_imgs = train_test_split(images, test_size=split_ratio, random_state=42)
            
            os.makedirs(os.path.join(TRAIN_DIR, label), exist_ok=True)
            os.makedirs(os.path.join(VAL_DIR, label), exist_ok=True)
            
            for img in train_imgs:
                shutil.copy2(os.path.join(src_path, img), os.path.join(TRAIN_DIR, label, img))
            for img in val_imgs:
                shutil.copy2(os.path.join(src_path, img), os.path.join(VAL_DIR, label, img))
            
            total_train += len(train_imgs)
            total_val += len(val_imgs)
            print(f'  ✅ {label}: {len(train_imgs)} train, {len(val_imgs)} val')
        
        print(f'\n✅ Split complete!')
        print(f'   Total: {total_train} train + {total_val} val = {total_train + total_val} images')
    else:
        raise ValueError(
            f'No class folders with images found.\n'
            f'Contents of {DATASET_PATH}: {contents}\n'
            f'Please set TRAIN_DIR and VAL_DIR manually.'
        )

print(f'\nTRAIN_DIR = {TRAIN_DIR}')
print(f'VAL_DIR   = {VAL_DIR}')
train_classes = sorted(os.listdir(TRAIN_DIR))
print(f'Train classes ({len(train_classes)}): {train_classes}')

## 3️⃣ Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt
import random
from PIL import Image

classes = sorted(os.listdir(TRAIN_DIR))
n_show = min(len(classes), 12)
fig, axes = plt.subplots(2, min(n_show, 6), figsize=(18, 6))
axes = axes.flatten()

for i, cls in enumerate(classes[:n_show]):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        img_path = os.path.join(cls_dir, random.choice(imgs))
        img = Image.open(img_path).resize((224, 224))
        axes[i].imshow(img)
        axes[i].set_title(cls.replace('___', '\n').replace('_', ' ')[:30], fontsize=8)
    axes[i].axis('off')

for j in range(n_show, len(axes)):
    axes[j].axis('off')

plt.suptitle('Sample Images from Each Class', fontsize=14)
plt.tight_layout()
plt.show()

# Print class distribution
print('\nClass Distribution:')
for cls in classes:
    count = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    print(f'  {cls}: {count} images')

## 4️⃣ Build Data Generators & CNN Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

# ---------- Configuration ----------
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_PHASE1 = 10   # Frozen base training
EPOCHS_PHASE2 = 15   # Fine-tuning
LEARNING_RATE = 0.0001

# ---------- Data Generators ----------
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255)

print('\nLoading training data...')
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

print('Loading validation data...')
val_gen = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

num_classes = train_gen.num_classes
class_labels = {v: k for k, v in train_gen.class_indices.items()}

print(f'\n✅ Classes found: {num_classes}')
print(f'   Training samples:   {train_gen.samples}')
print(f'   Validation samples: {val_gen.samples}')
print(f'\nClass names: {list(train_gen.class_indices.keys())}')

In [ ]:
# ---------- Build MobileNetV2 Model ----------
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze base model
base_model.trainable = False

# Custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print model summary
print(f'Total layers: {len(model.layers)}')
print(f'Trainable parameters: {sum(p.numpy().size for p in model.trainable_weights):,}')
print(f'Non-trainable parameters: {sum(p.numpy().size for p in model.non_trainable_weights):,}')

## 5️⃣ Train — Phase 1: Frozen Base (Feature Extraction)

In [ ]:
MODEL_SAVE_PATH = '/content/crop_disease_model.h5'

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print('🔹 Phase 1: Training classification head (base frozen)...')
print('=' * 60)

history1 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE1,
    validation_data=val_gen,
    callbacks=callbacks
)

phase1_acc = max(history1.history['val_accuracy'])
print(f'\n✅ Phase 1 complete. Best val accuracy: {phase1_acc:.4f}')

## 6️⃣ Train — Phase 2: Fine-Tuning Top Layers

In [ ]:
# Unfreeze top 30 layers of base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE / 10),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fine_tune_trainable = sum(p.numpy().size for p in model.trainable_weights)
print(f'Fine-tuning trainable parameters: {fine_tune_trainable:,}')
print()
print('🔹 Phase 2: Fine-tuning top layers...')
print('=' * 60)

history2 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE1 + EPOCHS_PHASE2,
    initial_epoch=len(history1.history['loss']),
    validation_data=val_gen,
    callbacks=callbacks
)

phase2_acc = max(history2.history['val_accuracy'])
print(f'\n✅ Phase 2 complete. Best val accuracy: {phase2_acc:.4f}')

## 7️⃣ Evaluate & Visualize Results

In [ ]:
# Final evaluation
print('📊 Final Evaluation')
print('=' * 40)
loss, accuracy = model.evaluate(val_gen)
print(f'\nValidation Loss:     {loss:.4f}')
print(f'Validation Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)')

In [ ]:
import matplotlib.pyplot as plt

# Combine histories
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss_vals = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(acc, label='Train Accuracy', linewidth=2)
ax1.plot(val_acc, label='Val Accuracy', linewidth=2)
ax1.axvline(x=len(history1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Model Accuracy', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(loss_vals, label='Train Loss', linewidth=2)
ax2.plot(val_loss, label='Val Loss', linewidth=2)
ax2.axvline(x=len(history1.history['loss'])-1, color='gray', linestyle='--', label='Fine-tune start')
ax2.set_title('Model Loss', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print('Training curves saved to /content/training_curves.png')

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Get predictions on validation set
val_gen.reset()
preds = model.predict(val_gen, verbose=1)
pred_classes = np.argmax(preds, axis=1)
true_classes = val_gen.classes
class_names = list(val_gen.class_indices.keys())

# Classification report
print('\n📋 Classification Report:')
print('=' * 60)
print(classification_report(true_classes, pred_classes, target_names=class_names))

# Confusion matrix (only if < 20 classes for readability)
if len(class_names) <= 20:
    cm = confusion_matrix(true_classes, pred_classes)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Greens', ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.savefig('/content/confusion_matrix.png', dpi=150)
    plt.show()

## 8️⃣ Save Model & Class Labels

In [ ]:
import json

# Save class labels
LABELS_PATH = '/content/class_labels.json'

# Convert int keys to str for JSON serialization
class_labels_str = {str(k): v for k, v in class_labels.items()}

with open(LABELS_PATH, 'w') as f:
    json.dump(class_labels_str, f, indent=2)

# Also save to Google Drive for backup
DRIVE_OUTPUT = '/content/drive/MyDrive/CropDisease_Model'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

import shutil
shutil.copy2(MODEL_SAVE_PATH, os.path.join(DRIVE_OUTPUT, 'crop_disease_model.h5'))
shutil.copy2(LABELS_PATH, os.path.join(DRIVE_OUTPUT, 'class_labels.json'))

print('✅ Files saved:')
print(f'   Model:  {MODEL_SAVE_PATH}')
print(f'   Labels: {LABELS_PATH}')
print(f'\n📁 Backup saved to Google Drive: {DRIVE_OUTPUT}')
print()
print('Class labels mapping:')
for idx, name in sorted(class_labels.items(), key=lambda x: x[0]):
    print(f'   {idx}: {name}')

## 9️⃣ Download Model Files to Your Computer

Run the cell below to download the 2 files. Then place them in your project:

```
crop-advisory-system/
  backend/
    ai_model/
      crop_disease_model.h5    ← place here
      class_labels.json        ← place here
```

The Flask app will automatically detect and use the trained model!

In [ ]:
from google.colab import files

print('Downloading crop_disease_model.h5...')
files.download(MODEL_SAVE_PATH)

print('Downloading class_labels.json...')
files.download(LABELS_PATH)

print('\n✅ Download complete!')
print('\nPlace both files in: crop-advisory-system/backend/ai_model/')
print('Then restart your Flask app — the CNN model will be loaded automatically.')

## 🔟 Test Model on a Sample Image

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt

# Pick a random validation image
import random
val_classes_dirs = os.listdir(VAL_DIR)
rand_class = random.choice(val_classes_dirs)
rand_images = os.listdir(os.path.join(VAL_DIR, rand_class))
rand_img = random.choice(rand_images)
test_path = os.path.join(VAL_DIR, rand_class, rand_img)

# Predict
img = load_img(test_path, target_size=(224, 224))
img_array = img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

preds = model.predict(img_array, verbose=0)
top5 = np.argsort(preds[0])[::-1][:5]

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title(f'True: {rand_class}', fontsize=11)
plt.axis('off')

plt.subplot(1, 2, 2)
names = [class_labels[i] for i in top5]
scores = [preds[0][i] for i in top5]
colors = ['green' if names[j] == rand_class else 'gray' for j in range(len(names))]
plt.barh(range(len(names)), scores, color=colors)
plt.yticks(range(len(names)), [n[:25] for n in names], fontsize=9)
plt.xlabel('Confidence')
plt.title('Top 5 Predictions')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

print(f'\nTrue label:      {rand_class}')
print(f'Predicted label: {class_labels[top5[0]]} ({preds[0][top5[0]]*100:.1f}%)')
print(f'Correct: {"✅" if class_labels[top5[0]] == rand_class else "❌"}')